In [ ]:
import os
import pandas as pd
import s3fs
import sklearn

In [ ]:
sklearn.__version__

In [ ]:
os.environ['AWS_S3_ENDPOINT']

In [ ]:
S3_ENDPOINT_URL = 'http://' + os.environ['AWS_S3_ENDPOINT']
S3_ENDPOINT_URL

In [ ]:
fs = s3fs.S3FileSystem(client_kwargs = {'endpoint_url' :S3_ENDPOINT_URL })

In [ ]:
fs.ls('')

In [ ]:
fs.ls('ematzner-ensae')

In [ ]:
BUCKET = 'ematzner-ensae'
FILE_KEY_S3 = '/readmission_avc.parquet'
FILE_PATH_S3 = BUCKET+FILE_KEY_S3

In [ ]:
FILE_PATH_S3

In [ ]:
with fs.open(FILE_PATH_S3, mode = "rb") as file_in : 
    dataini = pd.read_parquet(file_in)

In [ ]:
dataini.dtypes

# 1. Data Preprocessing

In [ ]:
dataini.isna().sum()

In [ ]:
dataini = dataini.dropna(subset=['id_D'], axis= 0).copy()

In [ ]:
dataini["rea"] = (dataini["id_D"] != "").astype("int8") # int8 ? 1 octet par valeur
# convertir en boolean

In [ ]:
dataini

In [ ]:
dataini.rea.value_counts()

In [ ]:
dataini.dtypes

In [ ]:
str_cols = ['modeEntree', 'modeSortie', 'sexe']


In [ ]:
# contre exemple
# dataini[str_cols] = dataini[str_cols].astype('str')

In [ ]:
dataini.isna().sum()

In [ ]:
dataini[str_cols] = dataini[str_cols].astype("object")

In [ ]:
dataini.isna().sum()

In [ ]:
dataini.nbda.value_counts()

In [ ]:
dataini['nbda'] = dataini['nbda'].fillna(0)

In [ ]:
dataset = dataini[dataini['modeSortie'] != 9]

In [ ]:
dataset = dataset.drop(['id', 'id_D'], axis = 1)

In [ ]:
dataset.to_parquet('dataset.parquet')

In [ ]:
# on suppose que fs est d?j? cr?? (s3fs.S3FileSystem)
OUT_PATH_S3 = BUCKET+"/dataini.parquet"

with fs.open(OUT_PATH_S3, mode="wb") as file_out:
    dataini.to_parquet(file_out, engine="pyarrow", index=False, compression="snappy")


In [ ]:
OUT_PATH_S3

# 2. Feature Engineering
## 2.1 Principes

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, Normalizer

# StandardScaler ? agit par variable (colonne), met chaque feature ? moyenne 0 et ?cart-type 1.

# Normalizer ? agit sur chaque individu (ligne).
# "Norme 1" ? le vecteur est r?duit ? une longueur 1, mais garde la m?me direction (il pointe dans le m?me sens).

In [ ]:
dataset['sexe']

In [ ]:
OneHotEncoder(sparse_output=False, drop='first').fit_transform(SimpleImputer(strategy= 'most_frequent').fit_transform(dataset[['sexe']]))

## 2.2 Preprocessing Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [ ]:
features = dataset.drop('rea', axis = 1)
label = dataset['rea']

In [ ]:
features.dtypes

In [ ]:
num_features = features.select_dtypes(include= ['int32', 'float64']).columns
cat_features = features.select_dtypes(include= ['object']).columns

In [ ]:
cat_features

In [ ]:
# ou
# cat_features = list(set(features.columns) - set(num_features))

In [ ]:
num_transformer = Pipeline(steps= [
    ('imputer', SimpleImputer()),
    ('scaler', StandardScaler())
])

In [ ]:
cat_transformer = Pipeline(steps= [
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown= 'ignore'))
])

In [ ]:
preprocessor = ColumnTransformer(transformers= [
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

In [ ]:
features.columns

In [ ]:
dataset.columns

In [ ]:
preprocessor.fit_transform(features).toarray()

In [ ]:
from sklearn.model_selection import train_test_split

X_train_val, X_test, y_train_val, y_test = train_test_split(features, label, test_size= 0.1, random_state= 18)

X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size= 0.1, random_state= 42)

In [ ]:
features.shape

In [ ]:
label.shape

In [ ]:
print(X_train.shape, X_val.shape, X_test.shape)
print(y_train.shape, y_val.shape, y_test.shape)

# 3. Model
## 3.1 Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
lr = LogisticRegression()

In [ ]:
pip_reg = Pipeline(steps=[
    ('preproc', preprocessor),
    ('classifier', lr)
])

In [ ]:
pip_reg_fitted = pip_reg.fit(X_train, y_train)

In [ ]:
y_predict = pip_reg_fitted.predict(X_val)

In [ ]:
y_predict_proba = pip_reg_fitted.predict_proba(X_val)[:,1]

In [ ]:
y_predict_proba

In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

In [ ]:
accuracy_score(y_val, y_predict)

In [ ]:
f1_score(y_val, y_predict)

In [ ]:
roc_auc_score(y_val, y_predict_proba)

## 3.2 RF

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from pprint import pprint

In [ ]:
rf = RandomForestClassifier(random_state= 42)

In [ ]:
pip_rf = Pipeline(steps=[
    ('preproc', preprocessor),
    ('classifier', rf)
]
)

In [ ]:
pip_rf_fitted = pip_rf.fit(X_train, y_train)

In [ ]:
rf_y_predict = pip_rf_fitted.predict(X_val)

In [ ]:
rf_y_predict_proba = pip_rf_fitted.predict_proba(X_val)[:,1]

In [ ]:
accuracy_score(y_val, rf_y_predict)

In [ ]:
# fitting 2 fois et montrer que le r?sultat est diff?rent
roc_auc_score(y_val, rf_y_predict_proba)

In [ ]:
pprint(rf.get_params())

**n_estimators** (Nombre d'arbres dans la for?t)<br/>

Pourquoi le tuner ?<br/>
* Augmenter le nombre d'arbres am?liore souvent la performance du mod?le (r?duit la variance), mais au prix d'une augmentation du temps d'entra?nement et d'inf?rence.
* Par d?faut, 100 arbres suffisent souvent, mais augmenter ? 200 ou 500 peut ?tre utile pour des datasets complexes.
* Valeurs courantes : 50, 100, 200, 500.


**max_depth** (Profondeur maximale des arbres)<br/>
Description : Contr?le la profondeur maximale de chaque arbre.

Pourquoi le tuner ?<br/>
* Limiter la profondeur aide ? ?viter le surapprentissage (overfitting).
* Des arbres tr?s profonds capturent trop de bruit dans les donn?es, tandis que des arbres peu profonds risquent de sous-apprendre (underfitting).

Valeurs courantes :<br/>
* None (par d?faut, arbre enti?rement d?velopp?).
* Valeurs typiques : 5, 10, 20, en fonction de la taille des donn?es.


**max_features** (Nombre de features ? consid?rer lors de la division)<br/>
Description : Nombre de features ? ?chantillonner al?atoirement pour rechercher la meilleure division ? chaque split.<br/>
* 'sqrt' : 
* 'log2' : 
* Un entier : Un nombre fixe de features.<br/>

Pourquoi le tuner ?
* Moins de features ? chaque split r?duit la corr?lation entre les arbres, ce qui am?liore la diversit? de la for?t.
* Trop peu de features peuvent limiter la capacit? d'apprentissage des arbres.
* Valeurs courantes : 'sqrt', 'log2', 0.3, 0.5 (fraction des features).

**min_samples_leaf** (Nombre mininum d??chantillons par feuille)<br/>
Description : Nombre mininum d'?chantillons requis pour qu'un n?ud soit une feuille (Le nombre minimum d'individus (ou de lignes) requis pour qu?un n?ud soit consid?r? comme une feuille.).<br/>
Pourquoi le tuner ?<br/>
* Augmenter cette valeur force les feuilles ? contenir plus d??chantillons, ce qui r?duit le surapprentissage.<br/>
* Cela est particuli?rement utile pour les datasets d?s?quilibr?s ou bruit?s.<br/>

Valeurs courantes : 1 (par d?faut), 5, 10, 20.

In [ ]:
param_rf = {
    'classifier__n_estimators' : [100, 200, 500],
    'classifier__max_depth' : [10, 20, None], #
    # 'max_features' : ['sqrt', 'log2']
}

# La profondeur max de ton arbre ne d?pend pas du nombre de variables, mais du nombre d??chantillons, des crit?res de split et des param?tres (max_depth, min_samples_split, min_samples_leaf)

### Effet de l?hyperparam?tre `max_features` avec 10 variables

- **`max_features='sqrt'`**  
  - On prend la racine carr?e du nombre total de variables.  
  - Ici : sqrt(10) ? 3.16 ? donc **3 variables** consid?r?es ? chaque split.  

- **`max_features='log2'`**  
  - On prend le logarithme en base 2 du nombre total de variables.  
  - Ici : log2(10) ? 3.32 ? donc **3 variables** aussi.  

?? Avec 10 variables, `sqrt` et `log2` donnent **le m?me r?sultat**.


In [ ]:
metric_grid = ['accuracy', 'f1', 'roc_auc']

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

In [ ]:
# StratifiedKFold agit uniquement sur la r?partition de la variable cible Y (les ?tiquettes de classes)

cv = StratifiedKFold(
    n_splits= 5,
    shuffle=True, # les donn?es sont m?lang?es al?atoirement avant d??tre d?coup?es en folds.
    random_state=42, 
    
)


# En pratique : 
# Dataset grand et ?quilibr? ? StratifiedKFold simple suffit.
# Dataset petit ou instable ? RepeatedStratifiedKFold est recommand?.

from sklearn.model_selection import RepeatedStratifiedKFold

# cv = RepeatedStratifiedKFold(
#     n_splits=5,   # nombre de folds
#     n_repeats=3,  # combien de fois on refait le CV avec des splits diff?rents
#     random_state=42
# )

In [ ]:
grid_rf = GridSearchCV(
    estimator= pip_rf,
    param_grid= param_rf,
    cv = cv,
    scoring = 'roc_auc', # metric selon lequel on d?termine le best_result,
    refit = True # par defaut
)

In [ ]:
grid_rf_fitted = grid_rf.fit(X_train_val, y_train_val)

In [ ]:
grid_rf_fitted.cv_results_.keys()

In [ ]:
grid_rf_fitted.best_params_

In [ ]:
grid_rf_fitted.best_score_

In [ ]:
grid_rf_fitted.score(X_test, y_test)

In [ ]:
# equivalent ?
roc_auc_score(y_test, grid_rf_fitted.best_estimator_.predict_proba(X_test)[:,1])

In [ ]:
# equivalent ?
# roc_auc_score(y_test, grid_rf_fitted.predict_proba(X_test)[:,1])

## 3.3 KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
pprint(KNeighborsClassifier().get_params())

#### Hyperparam?tres courants ? tuner pour KNN<br/>

**n_neighbors** (Nombre de voisins)<br/>
Description : D?finit combien de voisins sont consid?r?s pour prendre une d?cision.<br/>
Pourquoi le tuner ?<br/>
* Un nombre de voisins trop faible peut rendre le mod?le sensible au bruit (overfitting).
* Un nombre trop ?lev? peut le rendre trop g?n?ralisant (underfitting).
<br/>Valeurs courantes : 3, 5, 7, 10, 15.

**weights** (Pond?ration des voisins)
Description : D?finit si tous les voisins contribuent ?galement ou si leur contribution est pond?r?e par la distance.
'uniform' : Tous les voisins ont le m?me poids.
'distance' : Les voisins proches ont plus de poids que les voisins ?loign?s.
Pourquoi le tuner ? <br/>
La pond?ration par distance peut am?liorer la performance si les observations proches sont plus repr?sentatives de la classe cible.
Valeurs courantes : 'uniform', 'distance'.


**metric** (Distance utilis?e pour trouver les voisins)
Description : D?finit la mesure de distance utilis?e pour calculer la proximit? entre les points.
* 'minkowski' : Distance g?n?ralis?e.
* 'euclidean' : Distance euclidienne
* 'manhattan' : Distance Manhattan

Pourquoi le tuner ?<br/>
Certaines mesures de distance peuvent mieux capturer la structure des donn?es en fonction du probl?me.
Valeurs courantes : 'euclidean', 'manhattan', 'minkowski'.

**p** (Param?tre pour la distance de Minkowski)
Description : D?finit la puissance pour la distance de Minkowski.
* p=1 correspond ? la distance Manhattan.
* p=2 correspond ? la distance Euclidienne.
  
<br/>Pourquoi le tuner ?<br/>
* Ajuster la valeur de p permet d'explorer diff?rents types de relations dans les donn?es.
* Valeurs courantes : 1, 2, 3.


**algorithm** (Algorithme utilis? pour la recherche des voisins)
Description : D?finit la m?thode utilis?e pour rechercher les voisins les plus proches.
- 'auto' : Choisit automatiquement l'algorithme le plus adapt?.
- 'ball_tree' : Utilise une structure BallTree.
- 'kd_tree' : Utilise une structure KDTree.
- 'brute' : Recherche brute.
  
Pourquoi le tuner ?<br/>
Bien que 'auto' fonctionne dans la plupart des cas, certaines structures (comme kd_tree) peuvent ?tre plus rapides pour des donn?es sp?cifiques.



In [ ]:
knn = KNeighborsClassifier()

In [ ]:
pip_knn = Pipeline(steps=[
    ('preproc' , preprocessor),
    ('classifier' , knn)
    
])

In [ ]:
param_knn = {
    'preproc__num__scaler' : [StandardScaler(), MinMaxScaler(), Normalizer()],
    'classifier__n_neighbors' : [5, 7, 10,],
    'classifier__weights' : ['uniform', 'distance'],
    'classifier__metric' : ['euclidean', 'manhattan']
}

In [ ]:
grid_knn = GridSearchCV(
    estimator=pip_knn,
    param_grid=param_knn,
    cv = cv,
    scoring = 'roc_auc',
    refit = True
)

In [ ]:
grid_knn_fitted = grid_knn.fit(X_train_val, y_train_val)

In [ ]:
grid_knn_fitted.best_estimator_

In [ ]:
grid_knn_fitted.best_score_

In [ ]:
grid_knn_fitted.best_params_

In [ ]:
grid_knn_fitted.score(X_test, y_test)

In [ ]:
roc_auc_score(y_test, grid_knn_fitted.best_estimator_.predict_proba(X_test)[:,1])

## 3.4 Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

In [ ]:
gb = GradientBoostingClassifier()

In [ ]:
pprint(gb.get_params())

**learning_rate** <br/>
Description : Le Gradient Boosting construit les arbres s?quentiellement : chaque nouvel arbre est form? pour corriger les erreurs faites par les arbres pr?c?dents. ? chaque ?tape, le nouvel arbre ajuste les r?sidus (erreurs) des pr?dictions pr?c?dentes.Le learning_rate contr?le la contribution de chaque nouvel arbre ? la pr?diction globale ?? ?? ( ?? )<br/>
Importance : Tr?s ?lev?.<br/>
* Un taux d'apprentissage faible (ex. : 0.01) n?cessite plus d'arbres (n_estimators) pour atteindre des performances optimales.
* Un taux ?lev? (ex. : 0.2 ou 0.3) permet une convergence rapide, mais risque de sur-apprendre.
* Valeur par d?faut : 0.1.
* Valeurs courantes : [0.01, 0.05, 0.1, 0.2].


**n_estimators**<br/>
Description : Nombre total d'arbres dans le mod?le. A chaque it?ration, un seul arbre est ajout? au mod?le<br/>
Importance : Tr?s ?lev?.
* Un nombre insuffisant d'arbres peut sous-apprendre, tandis qu'un nombre trop ?lev? peut augmenter le temps d'entra?nement sans am?lioration significative.
* Valeur par d?faut : 100.
* Valeurs courantes : [100, 200, 500, 1000], d?pendant de la taille des donn?es et de learning_rate.

**max_depth**<br/>
Description : Profondeur maximale de chaque arbre.<br/>
Importance : ?lev?e.
* Limiter la profondeur emp?che les arbres de surapprendre en capturant trop de bruit.
* Une profondeur trop faible peut sous-apprendre les relations complexes dans les donn?es.
* Valeur par d?faut : 3.
* Valeurs courantes : [3, 5, 7, None].

**min_samples_split**<br/>
Description : Nombre mininum d'?chantillons requis pour diviser un n?ud.<br/>
Importance : Moyen.<br/>
* Permet de contr?ler la taille minimale des n?uds parents, ce qui limite les arbres tr?s complexes.
* Valeur par d?faut : 2.
* Valeurs courantes : [2, 5, 10].

**min_samples_leaf** <br/>
Description : Nombre mininum d'?chantillons requis pour ?tre dans une feuille.<br/>
Importance : Moyen.<br/>
* Emp?che les arbres de cr?er des feuilles contenant un tr?s petit nombre de donn?es, ce qui peut les rendre sensibles au bruit.
* Valeur par d?faut : 1.
* Valeurs courantes : [1, 5, 10, 20].

In [ ]:
gb = GradientBoostingClassifier(random_state= 42)

In [ ]:
pip_gb = Pipeline(steps = [
    ('preproc', preprocessor),
    ('classifier', gb)
])

In [ ]:
param_gb = {
    'classifier__n_estimators' : [200, 500, ],
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__max_depth' : [7, 10]
}

In [ ]:
grid_gb = GridSearchCV(
    estimator=pip_gb,
    cv = cv,
    param_grid=param_gb,
    scoring = 'roc_auc'
)

In [ ]:
grid_gb_fitted = grid_gb.fit(X_train_val, y_train_val)

In [ ]:
grid_gb_fitted.best_estimator_

In [ ]:
grid_gb_fitted.best_score_

In [ ]:
grid_gb_fitted.best_params_

In [ ]:
grid_gb_fitted.score(X_test, y_test)

In [ ]:
roc_auc_score(y_test, grid_gb_fitted.best_estimator_.predict_proba(X_test)[:, 1])

## 3.5 MLP

In [ ]:
from sklearn.neural_network import MLPClassifier

In [ ]:
mlp = MLPClassifier(random_state=42,
                   max_iter= 1000)

# ?? Synth?se MLPClassifier (scikit-learn)

### 1. Initialisation et al?atoire
- Les poids et biais du r?seau sont initialis?s al?atoirement.  
- `random_state` rend l?initialisation (et le shuffle des donn?es) **reproductible**.  

---

### 2. Epochs, it?rations et batchs
- **Mini-batch** : sous-ensemble du dataset utilis? pour une mise ? jour des poids.  
- **It?ration (step)** = 1 forward + 1 backward + 1 update sur **1 mini-batch**.  
- **Epoch** = passage complet sur tout le dataset ? donc plusieurs it?rations.  
- Relation :   nb_iterations = nb_epochs ? nb_batchs_par_epoch


### 3. Param?tre `max_iter`
- Dans `MLPClassifier`, `max_iter` = **nombre maximum d?epochs**.  
- Ce n?est pas un hyperparam?tre de tuning, mais une borne sup?rieure.  
- Exemple :  
- Dataset = 1000 exemples, `batch_size=100` ? 10 batchs/epoch  
- `max_iter=200` ? 200 epochs max ? 200 ? 10 = 2000 updates des poids.  

---

### 4. Convergence et arr?t
- Le mod?le peut s?arr?ter **avant `max_iter`** si la convergence est atteinte.  
- Crit?re d?arr?t :  
- `tol` : tol?rance sur l?am?lioration minimale,  
- `n_iter_no_change` : nb d?epochs cons?cutifs sans am?lioration.  
- Avec `early_stopping=True`, l?arr?t se fait sur un jeu de validation.  

---

### 5. Hyperparam?tres importants ? tuner
- `hidden_layer_sizes` : architecture (nb de couches / nb de neurones).  
- `alpha` : r?gularisation L2.  
- `learning_rate_init` : taux d?apprentissage.  
- `activation` : fonction d?activation (`relu`, `tanh`, `logistic`).  
- `solver` : m?thode d?optimisation (`adam`, `sgd`, `lbfgs`).  
- (Parfois `batch_size`).  

---

### 6. R?sum?
- **It?ration** = 1 batch ? 1 update.  
- **Epoch** = tous les batchs ? plusieurs updates.  
- **max_iter** = nb max d?epochs (donc plusieurs it?rations).  
- L?entra?nement peut s?arr?ter avant convergence.


In [ ]:
pip_mlp = Pipeline(
    steps=[
        ('preproc', preprocessor),
        ('classifier', mlp)
    ]
)

In [ ]:
param_mlp = {
    'classifier__hidden_layer_sizes': [(50,), (100,), (50, 50)],
    'classifier__activation': ['relu', 'tanh'],
    'classifier__solver': ['adam', 'sgd'], # l'algorithme d'optimisation utilis? pour mettre ? jour les poids lors de l'entra?nement du r?seau de neurones
    'classifier__alpha': [0.0001, 0.001], # dans sklearn, r?gularisation est toujours ridge, alpha correspond ? l'intensit? de la p?nalit?
}

In [ ]:
grid_mlp = GridSearchCV(
    estimator= pip_mlp,
    param_grid= param_mlp,
    cv = cv,
    scoring= 'roc_auc'
)

In [ ]:
grid_mlp_fitted = grid_mlp.fit(X_train_val, y_train_val)

# 4. Model selection Pipeline 

In [ ]:
models = [
    # ('lr', lr, {'classifier__C': [1.0]} ), # inverse de lambda, ?=1 ? r?gularisation "standard"
    # ('rf', rf, param_rf),
    # ('knn', knn, param_knn),
    ('gb', gb, param_gb),
]

In [ ]:
preproc_list = [
    # {'id' : 'basic', 'object': preprocessor},
    # {'id' : 'interact', 'object': preproc_interact},
    {'id' : 'spline', 'object': preproc_spline},
]

In [ ]:
results = []
for preproc in preproc_list : 
    preproc_id = preproc['id']
    preproc_object = preproc['object']
    for model_id, model_object, param in models : 
        print(f'Model {model_id} with preprocessor {preproc_id}')
        pip = Pipeline(steps= [
            ('preproc', preproc_object),
            ('classifier', model_object)
        ])

        grid = GridSearchCV(
            estimator= pip,
            cv=cv, 
            param_grid = param,
            scoring= metric_grid,
            refit = 'roc_auc'
        ).fit(X_train_val, y_train_val)

        results.append(
            {
                'preprocessor' : preproc_id,
                'model' : model_id,
                'best_param' : grid.best_params_,
                'best_score' : grid.best_score_,
                'final_prediction' : grid.score(X_test, y_test)
            }
        )        

In [ ]:
pd.DataFrame(results)

# 5. More Feature Engineering
## 5.1 interaction

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, SplineTransformer

In [ ]:
import numpy as np

In [ ]:
X = np.arange(6).reshape(3,2)
X

In [ ]:
poly = PolynomialFeatures(2, interaction_only=True)
poly.fit_transform(X)

In [ ]:
interact_transformer = Pipeline(steps=[
    ('imp', SimpleImputer()),
    ('poly', PolynomialFeatures(interaction_only=True)),
    ('scaler', StandardScaler())
])

In [ ]:
fe_variables = ['age', 'duree', 'nbda']

In [ ]:
no_fe_variable = num_features.drop(fe_variables)

In [ ]:
preproc_interact = ColumnTransformer(transformers=[
    ('num', num_transformer, no_fe_variable),
    ('interact', interact_transformer, fe_variables),
    ('cat', cat_transformer, cat_features)
]
    
)

## 5.2 Spline

In [ ]:
spline_transformer = Pipeline(steps=[
    ('imp', SimpleImputer()),
    ('poly', SplineTransformer()),
    ('scaler', StandardScaler())
])

In [ ]:
preproc_spline = ColumnTransformer(transformers=[
    ('num', num_transformer, no_fe_variable),
    ('interact', spline_transformer, fe_variables),
    ('cat', cat_transformer, cat_features)
]   
)

## Pourquoi a-t-on besoin d?un jeu de test ?

### R?les des jeux de donn?es
- **Train set** : utilis? pour estimer les **param?tres** du mod?le (poids, coefficients, splits?).  
- **Validation set (ou CV interne)** : utilis? pour choisir les **hyperparam?tres** (ex. profondeur d?arbre, C en r?gression logistique, k en KNN).  
- **Test set** : totalement s?par?, jamais vu ? sert ? mesurer la performance r?elle du mod?le final.

---

### Point important
- Les **hyperparam?tres sont choisis en fonction du val set**.  
- Donc le **val set n?est pas neutre** : il influence directement le choix du mod?le.
- les hyperparam?tres marchent tr?s bien sur la validation, mais g?n?ralisent mal ? test set plus bas.

---

### R?le du test set
- Fournir une **?valuation impartiale et neutre**.  
- Comme il n?a jamais servi ni ? l?apprentissage, ni au choix des hyperparam?tres, il refl?te mieux la performance attendue en production.

---

### R?sum?
- **Train** ? apprend les param?tres.  
- **Val** ? choisit les hyperparam?tres (donc biais?).  
- **Test** ? seule vraie estimation neutre de la performance finale.


## Que faire si le score CV et le score Test sont diff?rents ?

### Situation typique
- Score CV (moyenne des folds internes) = **0.85**  
- Score sur test = **0.75**  
?cart important ? le mod?le g?n?ralise mal.

---

### Causes possibles
| Cause | Signes | Solutions |
|-------|--------|-----------|
| **Sur-optimisation (overfitting sur le val set)** | CV tr?s bon mais test beaucoup plus bas | R?duire l?espace d?hyperparam?tres, simplifier le mod?le, r?gulariser |
| **CV bruit?e (instable)** | Scores CV tr?s variables entre folds | Augmenter le nombre de folds (`cv=10`), utiliser `RepeatedStratifiedKFold`, v?rifier la variance |
| **Test set diff?rent (distribution shift)** | Test issu d?une autre source, p?riode ou population | V?rifier la repr?sentativit? du split, collecter plus de donn?es, envisager une adaptation de domaine |

---

### R?gle d?or
- **?cart faible** (ex. CV=0.82, Test=0.80) ? normal, coh?rent.  
- **?cart fort** (ex. CV=0.85, Test=0.75) ? probl?me de sur-tuning, de bruit, ou de distribution non repr?sentative.  

Ne jamais utiliser le **test set** pour retuner les hyperparam?tres ? il doit rester **neutre** jusqu?? la fin.
